# Convert Alice EEG Dataset to BIDS

This notebook converts the raw Alice EEG dataset in BrainVision format into BIDS EEG format.

Source dataset:
- `/Users/yanyuwoo/Data/r`

Output BIDS dataset:
- `/Users/yanyuwoo/Data/bids`

Run the cells from top to bottom.

(This script already executed, move to this project for reference)

## 1. Imports

If `mne` or `mne-bids` is missing, install them in the notebook kernel environment first.

In [1]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import json

import mne
from mne_bids import BIDSPath, write_raw_bids
from scipy.io import loadmat

## 2. Configure input and output paths

In [2]:
RAW_DATA_ROOT = Path('/Users/yanyuwoo/Data/r')
BIDS_ROOT = Path('/Users/yanyuwoo/Data/bids')
PROC_ZIP = RAW_DATA_ROOT / 'proc.zip'

print('Raw data root:', RAW_DATA_ROOT)
print('BIDS output root:', BIDS_ROOT)
print('Historical preprocessing metadata:', PROC_ZIP)
print('Raw data exists:', RAW_DATA_ROOT.exists())
print('BIDS output exists:', BIDS_ROOT.exists())
print('Historical metadata exists:', PROC_ZIP.exists())

Raw data root: /Users/yanyuwoo/Data/r
BIDS output root: /Users/yanyuwoo/Data/bids
Historical preprocessing metadata: /Users/yanyuwoo/Data/r/proc.zip
Raw data exists: True
BIDS output exists: True
Historical metadata exists: True


## 3. Check that the BrainVision files are present

In [4]:
subjects = [f'{i:02d}' for i in range(1, 50)]
missing = []

for subject in subjects:
    for ext in ('.vhdr', '.eeg', '.vmrk'):
        path = RAW_DATA_ROOT / f'S{subject}{ext}'
        if not path.exists():
            missing.append(str(path))

print('Subjects expected:', len(subjects))
print('Missing files:', len(missing))
missing[:10]

Subjects expected: 49
Missing files: 0


[]

## 4. Load historical bad-channel metadata

Bad-channel annotations are stored separately in `proc.zip`, under `proc.rejections.badchans`. A bad-channel name that is absent from the BrainVision recording (notably the implicit reference channel 29) is reported but cannot be written to BIDS `channels.tsv`.

In [5]:
def channel_sort_key(name):
    try:
        return 0, int(name)
    except ValueError:
        return 1, name


def normalize_channel_names(value):
    if isinstance(value, str):
        return [value]

    values = value.tolist() if hasattr(value, 'tolist') else list(value)
    if not isinstance(values, list):
        values = [values]
    return [str(name) for name in values]


def load_historical_bad_channels(proc_zip):
    bad_channels = {}
    proc_subjects = set()

    with ZipFile(proc_zip) as archive:
        proc_files = sorted(
            name
            for name in archive.namelist()
            if name.startswith('timelock-preprocessing/S')
            and name.endswith('.mat')
        )

        for filename in proc_files:
            mat = loadmat(
                BytesIO(archive.read(filename)),
                simplify_cells=True,
            )
            proc = mat['proc']
            subject = str(proc['subject']).removeprefix('S')
            proc_subjects.add(subject)
            bad_channels[subject] = normalize_channel_names(
                proc['rejections'].get('badchans', [])
            )

    for subject, names in bad_channels.items():
        bad_channels[subject] = sorted(set(names), key=channel_sort_key)

    return bad_channels, proc_subjects


historical_bad_channels, proc_subjects = load_historical_bad_channels(PROC_ZIP)
subjects_without_proc = sorted(set(subjects) - proc_subjects)

print('Subjects with proc metadata:', len(proc_subjects))
print('Subjects without proc metadata:', subjects_without_proc)
print('S01 historical bad channels:', historical_bad_channels['01'])

Subjects with proc metadata: 42
Subjects without proc metadata: ['28', '29', '31', '33', '46', '47', '49']
S01 historical bad channels: ['22', '24', '31', '32', '36', '49', '60']


## 5. Inspect one raw file before conversion

In [6]:
sample_vhdr = RAW_DATA_ROOT / 'S01.vhdr'
raw = mne.io.read_raw_brainvision(sample_vhdr, preload=False)
raw

Extracting parameters from /Users/yanyuwoo/Data/r/S01.vhdr...
Setting channel info structure...


<RawBrainVision | S01.eeg, 62 x 366525 (733.0 s), ~54 KiB, data not loaded>

## 6. Convert all subjects to BIDS

This will create a BIDS EEG dataset under `/Users/yanyuwoo/Data/bids`.

If you want to rerun the conversion from scratch, keep `overwrite=True`.

In [7]:
BIDS_ROOT.mkdir(parents=True, exist_ok=True)
conversion_qc = []

for subject in subjects:
    vhdr_file = RAW_DATA_ROOT / f'S{subject}.vhdr'
    print(f'Converting subject {subject}: {vhdr_file.name}')

    raw = mne.io.read_raw_brainvision(vhdr_file, preload=False)
    requested_bads = historical_bad_channels.get(subject, [])
    raw_channel_names = set(raw.ch_names)
    bads_in_raw = [name for name in requested_bads if name in raw_channel_names]
    bads_missing_from_raw = [name for name in requested_bads if name not in raw_channel_names]
    raw.info['bads'] = bads_in_raw

    print(f'  Bad channels written: {bads_in_raw}')
    if subject not in proc_subjects:
        print('  WARNING: no proc metadata; no historical bad-channel flags imported')
    if bads_missing_from_raw:
        print(f'  WARNING: bad-channel flags absent from raw: {bads_missing_from_raw}')

    bids_path = BIDSPath(
        subject=subject,
        task='alice',
        datatype='eeg',
        root=BIDS_ROOT,
    )

    write_raw_bids(
        raw,
        bids_path,
        overwrite=True,
        format='BrainVision',
        allow_preload=False,
    )

    conversion_qc.append({
        'subject': subject,
        'has_proc_metadata': subject in proc_subjects,
        'bad_channels_written': bads_in_raw,
        'bad_channels_absent_from_raw': bads_missing_from_raw,
    })

print('Done.')
print('Subjects converted:', len(conversion_qc))
print('Subjects without proc metadata:', subjects_without_proc)
print(
    'Historical bad-channel flags written:',
    sum(len(row['bad_channels_written']) for row in conversion_qc),
)
print(
    'Historical bad-channel flags absent from raw:',
    {
        row['subject']: row['bad_channels_absent_from_raw']
        for row in conversion_qc
        if row['bad_channels_absent_from_raw']
    },
)

Converting subject 01: S01.vhdr
Extracting parameters from /Users/yanyuwoo/Data/r/S01.vhdr...
Setting channel info structure...
  Bad channels written: ['22', '24', '31', '32', '36', '49', '60']
Extracting parameters from /Users/yanyuwoo/Data/r/S01.vhdr...
Setting channel info structure...
Writing '/Users/yanyuwoo/Data/bids/participants.tsv'...
Writing '/Users/yanyuwoo/Data/bids/participants.json'...
The provided raw data contains annotations, but you did not pass an "event_id" mapping from annotation descriptions to event codes. We will generate arbitrary event codes. To specify custom event codes, please pass "event_id".
Used Annotations descriptions: [np.str_('Stimulus/1'), np.str_('Stimulus/10'), np.str_('Stimulus/11'), np.str_('Stimulus/12'), np.str_('Stimulus/2'), np.str_('Stimulus/3'), np.str_('Stimulus/4'), np.str_('Stimulus/5'), np.str_('Stimulus/6'), np.str_('Stimulus/7'), np.str_('Stimulus/8'), np.str_('Stimulus/9')]
Writing '/Users/yanyuwoo/Data/bids/sub-01/eeg/sub-01_task-

/var/folders/j8/hc9f3z3931ld2qv6m5zzg0t80000gn/T/ipykernel_73950/342716130.py:28: RuntimeWarning: Converting data files to BrainVision format
  write_raw_bids(


Writing '/Users/yanyuwoo/Data/bids/sub-01/sub-01_scans.tsv'...
Wrote /Users/yanyuwoo/Data/bids/sub-01/sub-01_scans.tsv entry with eeg/sub-01_task-alice_eeg.vhdr.
Converting subject 02: S02.vhdr
Extracting parameters from /Users/yanyuwoo/Data/r/S02.vhdr...
Setting channel info structure...
  Bad channels written: ['10', '11', '46', '47', '61']
Extracting parameters from /Users/yanyuwoo/Data/r/S02.vhdr...
Setting channel info structure...
Writing '/Users/yanyuwoo/Data/bids/participants.tsv'...
Writing '/Users/yanyuwoo/Data/bids/participants.json'...
The provided raw data contains annotations, but you did not pass an "event_id" mapping from annotation descriptions to event codes. We will generate arbitrary event codes. To specify custom event codes, please pass "event_id".
Used Annotations descriptions: [np.str_('Stimulus/10'), np.str_('Stimulus/11'), np.str_('Stimulus/12'), np.str_('Stimulus/2'), np.str_('Stimulus/3'), np.str_('Stimulus/4'), np.str_('Stimulus/5'), np.str_('Stimulus/6'), 

/var/folders/j8/hc9f3z3931ld2qv6m5zzg0t80000gn/T/ipykernel_73950/342716130.py:28: RuntimeWarning: Converting data files to BrainVision format
  write_raw_bids(


Writing '/Users/yanyuwoo/Data/bids/sub-02/sub-02_scans.tsv'...
Wrote /Users/yanyuwoo/Data/bids/sub-02/sub-02_scans.tsv entry with eeg/sub-02_task-alice_eeg.vhdr.
Converting subject 03: S03.vhdr
Extracting parameters from /Users/yanyuwoo/Data/r/S03.vhdr...
Setting channel info structure...
  Bad channels written: ['10', '27', '31', '49', '50', '52', '54']
Extracting parameters from /Users/yanyuwoo/Data/r/S03.vhdr...
Setting channel info structure...
Writing '/Users/yanyuwoo/Data/bids/participants.tsv'...
Writing '/Users/yanyuwoo/Data/bids/participants.json'...
The provided raw data contains annotations, but you did not pass an "event_id" mapping from annotation descriptions to event codes. We will generate arbitrary event codes. To specify custom event codes, please pass "event_id".
Used Annotations descriptions: [np.str_('Stimulus/1'), np.str_('Stimulus/10'), np.str_('Stimulus/11'), np.str_('Stimulus/12'), np.str_('Stimulus/2'), np.str_('Stimulus/3'), np.str_('Stimulus/4'), np.str_('St

...

Writing '/Users/yanyuwoo/Data/bids/sub-48/eeg/sub-48_task-alice_events.json'...
Writing '/Users/yanyuwoo/Data/bids/dataset_description.json'...
Writing '/Users/yanyuwoo/Data/bids/sub-48/eeg/sub-48_task-alice_eeg.json'...
Copying data files to sub-48_task-alice_eeg.vhdr
Writing '/Users/yanyuwoo/Data/bids/sub-48/eeg/sub-48_task-alice_channels.tsv'...
Reading 0 ... 364224  =      0.000 ...   728.448 secs...
Writing '/Users/yanyuwoo/Data/bids/sub-48/sub-48_scans.tsv'...
Wrote /Users/yanyuwoo/Data/bids/sub-48/sub-48_scans.tsv entry with eeg/sub-48_task-alice_eeg.vhdr.
Converting subject 49: S49.vhdr
Extracting parameters from /Users/yanyuwoo/Data/r/S49.vhdr...
Setting channel info structure...
  Bad channels written: []
Extracting parameters from /Users/yanyuwoo/Data/r/S49.vhdr...
Setting channel info structure...
Writing '/Users/yanyuwoo/Data/bids/participants.tsv'...
Writing '/Users/yanyuwoo/Data/bids/participants.json'...
The provided raw data contains annotations, but you did not pass a

/var/folders/j8/hc9f3z3931ld2qv6m5zzg0t80000gn/T/ipykernel_73950/342716130.py:28: RuntimeWarning: Converting data files to BrainVision format
  write_raw_bids(


Writing '/Users/yanyuwoo/Data/bids/sub-49/eeg/sub-49_task-alice_events.json'...
Writing '/Users/yanyuwoo/Data/bids/dataset_description.json'...
Writing '/Users/yanyuwoo/Data/bids/sub-49/eeg/sub-49_task-alice_eeg.json'...
Copying data files to sub-49_task-alice_eeg.vhdr
Writing '/Users/yanyuwoo/Data/bids/sub-49/eeg/sub-49_task-alice_channels.tsv'...
Reading 0 ... 365949  =      0.000 ...   731.898 secs...


/var/folders/j8/hc9f3z3931ld2qv6m5zzg0t80000gn/T/ipykernel_73950/342716130.py:28: RuntimeWarning: Converting data files to BrainVision format
  write_raw_bids(


Writing '/Users/yanyuwoo/Data/bids/sub-49/sub-49_scans.tsv'...
Wrote /Users/yanyuwoo/Data/bids/sub-49/sub-49_scans.tsv entry with eeg/sub-49_task-alice_eeg.vhdr.
Done.
Subjects converted: 49
Subjects without proc metadata: ['28', '29', '31', '33', '46', '47', '49']
Historical bad-channel flags written: 294
Historical bad-channel flags absent from raw: {'26': ['29'], '48': ['29']}


## 7. Inspect the generated BIDS structure

In [8]:
top_level = sorted(p.name for p in BIDS_ROOT.iterdir())
top_level[:20]

['.DS_Store',
 'README',
 'dataset_description.json',
 'derivatives',
 'participants.json',
 'participants.tsv',
 'stimuli',
 'sub-01',
 'sub-02',
 'sub-03',
 'sub-04',
 'sub-05',
 'sub-06',
 'sub-07',
 'sub-08',
 'sub-09',
 'sub-10',
 'sub-11',
 'sub-12',
 'sub-13']

In [9]:
sub01_eeg = BIDS_ROOT / 'sub-01' / 'eeg'
sorted(p.name for p in sub01_eeg.iterdir())

['sub-01_task-alice_channels.tsv',
 'sub-01_task-alice_eeg.eeg',
 'sub-01_task-alice_eeg.json',
 'sub-01_task-alice_eeg.vhdr',
 'sub-01_task-alice_eeg.vmrk',
 'sub-01_task-alice_events.json',
 'sub-01_task-alice_events.tsv']

## 8. Optional: write a simple dataset description patch

Usually `write_raw_bids()` creates `dataset_description.json`. This cell lets you inspect it.

In [10]:
dataset_description = BIDS_ROOT / 'dataset_description.json'
print(dataset_description)
print(dataset_description.exists())

if dataset_description.exists():
    with open(dataset_description, 'r') as f:
        print(json.dumps(json.load(f), indent=2))

/Users/yanyuwoo/Data/bids/dataset_description.json
True
{
  "Name": "[Unspecified]",
  "BIDSVersion": "1.9.0",
  "DatasetType": "raw",
  "Authors": [
    "[Unspecified1]",
    "[Unspecified2]"
  ],
  "GeneratedBy": [
    {
      "Name": "MNE-BIDS",
      "Version": "0.18.0",
      "CodeURL": "https://mne.tools/mne-bids/"
    }
  ]
}


## Notes

- This notebook converts the raw EEG recordings to BIDS and imports historical bad-channel flags from `proc.zip`.
- Subjects without a matching `proc` file and bad-channel names absent from the raw recording are reported explicitly.
- It does not yet generate TRF predictors.
- After conversion, later analysis code should point to `/Users/yanyuwoo/Data/bids`, not to the old path from the error message.